# A/B Test Simulator Walkthrough
A compact tour of synthetic data, power, frequentist inference, sequential testing, and Bayesian decisions.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from abtest.bayesian import beta_binomial
from abtest.data import generate_experiment
from abtest.frequentist import conversion_ztest, srm_check
from abtest.power import analytic_sample_size, power_curve
from abtest.sequential import sequential_aa_simulation

## Generate an experiment
The generator includes skewed revenue, device and user-type segments, novelty decay, and optional SRM injection.

In [ ]:
data = generate_experiment(n_users=20_000, baseline_cr=0.04, relative_lift=0.05, seed=42)
data.head()

## Power means repeated experiments
Analytic power is a planning quantity; simulated power estimates how often the complete test rejects under a known effect.

In [ ]:
print('Users per arm:', analytic_sample_size(0.04, 0.05))
power_curve([1000, 3000, 5000], 0.04, 0.05, simulations=100)

In [ ]:
print(conversion_ztest(data))
print(srm_check(data))
print(beta_binomial(data, draws=10_000)['prob_treatment_better'])

## Peeking
A/A tests have no true treatment effect. Repeated daily looks inflate false positives; Bonferroni alpha splitting brings the error rate back toward the target.

In [ ]:
unfixed = sequential_aa_simulation(simulations=500, seed=42)
fixed = sequential_aa_simulation(simulations=500, correction='bonferroni', seed=42)
unfixed['false_positive_rate'], fixed['false_positive_rate']